# AgentSynth — fine-tune & benchmark (Colab)

Generate verified agent data with AgentSynth, fine-tune a small model on it, and
print a before/after function-calling benchmark. Runs on a **free Colab T4**.

First: **Runtime → Change runtime type → T4 GPU**.


## 1. Install


In [ ]:
%pip install -q "agentsynth[train,hub]" unsloth
# Not on PyPI yet? Install from git instead:
# %pip install -q "agentsynth[train,hub] @ git+https://github.com/agentsynth/agentsynth" unsloth


## 2. Generate a verified dataset (CPU, no API key needed)

Mock by default. To generate with a real LLM, set a provider key (e.g.
`os.environ['ANTHROPIC_API_KEY']`) and remove the force-mock line.


In [ ]:
import os

os.environ.setdefault('AGENTSYNTH_FORCE_MOCK', '1')

from agentsynth import (
    AgentTrajectoryGenerator,
    Recipe,
    TrajectoryEvaluator,
    build_dpo_dataset,
    build_preference_pairs,
    build_sft_dataset,
    run_recipe,
)

result = run_recipe(Recipe(num_trajectories=500, vary_modes=True,
                           verify=True, dedup=True, rubric='strict'))
build_sft_dataset(result.trajectories, 'sft.jsonl')
print('trajectories:', len(result.trajectories),
      '| pass@1:', result.metrics['pass_rate'],
      '| verified:', result.metrics.get('verified_rate'))

# preference pairs for DPO
gen, judge = AgentTrajectoryGenerator(), TrajectoryEvaluator()
queries = list({t.query for t in result.trajectories})[:150]
pairs = build_preference_pairs(gen, judge, queries, k=6)
build_dpo_dataset(pairs, 'dpo.jsonl')
print('dpo pairs:', len(pairs))


## 3. Load a small base model and benchmark it (before)


In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    'unsloth/Llama-3.2-1B-Instruct', max_seq_length=2048, load_in_4bit=True)

def hf_complete(prompt):
    FastLanguageModel.for_inference(model)
    ids = tokenizer.apply_chat_template(
        [{'role': 'user', 'content': prompt}], add_generation_prompt=True,
        return_tensors='pt').to(model.device)
    out = model.generate(input_ids=ids, max_new_tokens=128, do_sample=False)
    return tokenizer.decode(out[0][ids.shape[1]:], skip_special_tokens=True)

from agentsynth.benchmarks import prompted_model, report_table_md, run_benchmark

before = run_benchmark(prompted_model(hf_complete))
print('BEFORE  tool:', before.tool_accuracy, 'arg:', before.arg_accuracy, 'score:', before.score)


## 4. Supervised fine-tune (SFT)


In [ ]:
from datasets import load_dataset
from trl import SFTConfig, SFTTrainer

model = FastLanguageModel.get_peft_model(model, r=16, lora_alpha=16, lora_dropout=0.0)
sft_ds = load_dataset('json', data_files='sft.jsonl', split='train').select_columns(['messages'])

# trl >= 1.0 uses processing_class (older versions used tokenizer=)
trainer = SFTTrainer(
    model=model, processing_class=tokenizer, train_dataset=sft_ds,
    args=SFTConfig(output_dir='out/sft', max_steps=60, learning_rate=2e-4,
                   per_device_train_batch_size=2, gradient_accumulation_steps=4,
                   logging_steps=10))
trainer.train()


## 5. (optional) DPO on the preference pairs


In [ ]:
from trl import DPOConfig, DPOTrainer

dpo_ds = load_dataset('json', data_files='dpo.jsonl', split='train')
dpo = DPOTrainer(
    model=model, args=DPOConfig(output_dir='out/dpo', max_steps=40, learning_rate=5e-6,
                                beta=0.1, per_device_train_batch_size=2,
                                gradient_accumulation_steps=4, logging_steps=10),
    train_dataset=dpo_ds, processing_class=tokenizer)
dpo.train()


## 6. Benchmark again (after) and print the before/after table


In [ ]:
after = run_benchmark(prompted_model(hf_complete))
comparison = {
    'before': before, 'after': after, 'n': before.n,
    'delta_tool_accuracy': round(after.tool_accuracy - before.tool_accuracy, 4),
    'delta_score': round(after.score - before.score, 4),
}
print(report_table_md(comparison))


## 7. (optional) Publish the dataset to the Hugging Face Hub


In [ ]:
# from huggingface_hub import login; login()
# from agentsynth import push_dataset
# url = push_dataset(result.trajectories, 'your-username/agentsynth-trajectories',
#                    eval_results=result.eval_results)
# print(url)
